In [6]:
# pip install chembl_webresource_client pandas numpy
from chembl_webresource_client.new_client import new_client
import pandas as pd
import numpy as np
from math import log10

# --- 1) (Optional) Look up a target ID by name keyword ---
def find_targets(query: str, organism="Homo sapiens", limit=10):
    t = new_client.target
    hits = t.search(query)
    rows = []
    for h in hits:
        if organism and h.get("organism") != organism:
            continue
        rows.append(
            {
                "target_chembl_id": h["target_chembl_id"],
                "pref_name": h.get("pref_name"),
                "organism": h.get("organism"),
                "target_type": h.get("target_type"),
            }
        )
        # if len(rows) >= limit:
        #     break
    return pd.DataFrame(rows)

print(find_targets("EGFR"))

   target_chembl_id                                          pref_name  \
0     CHEMBL4523747                                        EGFR/PPP1CA   
1     CHEMBL5465557                                          CCN2-EGFR   
2         CHEMBL203                   Epidermal growth factor receptor   
3     CHEMBL4523680  Protein cereblon/Epidermal growth factor receptor   
4     CHEMBL2111431  Epidermal growth factor receptor and ErbB2 (HE...   
5     CHEMBL2363049                   Epidermal growth factor receptor   
6     CHEMBL4523998    von Hippel-Lindau disease tumor suppressor/EGFR   
7     CHEMBL4630723                          ErbB-2/ErbB-3 heterodimer   
8        CHEMBL1824            Receptor tyrosine-protein kinase erbB-2   
9        CHEMBL3009            Receptor tyrosine-protein kinase erbB-4   
10       CHEMBL5838            Receptor tyrosine-protein kinase erbB-3   
11    CHEMBL3137284  MER intracellular domain/EGFR extracellular do...   
12    CHEMBL4106134                   

In [7]:
targets = find_targets("EGFR")
print(targets.shape)

(17, 4)


In [16]:
def fetch_potencies_for_target(target_chembl_id: str,
                               types=("IC50","Ki","Kd","EC50"),
                               min_conf=8):
    """
    Returns a DataFrame of activities for the target with key fields.
    Filters:
      - standard_type in types
      - assay_confidence_score >= min_conf (if present)
      - drops rows with non-numeric standard_value or missing units/type
    """
    act = new_client.activity
    # Pull fields we’ll actually use
    fields = [
        "activity_id", "assay_chembl_id", "assay_type", "assay_confidence_score",
        "molecule_chembl_id",
        "standard_type", "standard_value", "standard_units", "relation",
        "pchembl_value", "target_chembl_id"
    ]
    q = act.filter(
        target_chembl_id=target_chembl_id,
        standard_type__in=list(types)
    ).only(fields)

    # Materialize (handles pagination internally)
    data = list(q)
    return data
    # if not data:
    #     return pd.DataFrame(columns=fields)
    #
    # df = pd.DataFrame(data)
    #
    # # Basic cleaning
    # # keep numeric standard_value
    # df = df[pd.to_numeric(df["standard_value"], errors="coerce").notna()].copy()
    # df["standard_value"] = df["standard_value"].astype(float)
    #
    # # filter by confidence if column present
    # if "assay_confidence_score" in df.columns and min_conf is not None:
    #     df = df[pd.to_numeric(df["assay_confidence_score"], errors="coerce").fillna(0) >= min_conf]
    #
    # # keep binding/functional if present (B/F)
    # if "assay_type" in df.columns:
    #     df = df[df["assay_type"].isin(["B", "F"])]
    #
    # # drop qualifiers like <, > for MVP
    # if "relation" in df.columns:
    #     df = df[df["relation"].isin(["=", "~"])]
    #
    # # --- 3) Compute pChEMBL if missing (IC50/Ki/Kd/EC50 in nM preferred) ---
    # def pchembl_from_row(r):
    #     if pd.notna(r.get("pchembl_value")):
    #         return float(r["pchembl_value"])
    #     # try to compute: if units are nM, pChEMBL = 9 - log10(nM)
    #     units = (r.get("standard_units") or "").lower()
    #     val = r.get("standard_value")
    #     if pd.isna(val):
    #         return np.nan
    #     try:
    #         val = float(val)
    #     except Exception:
    #         return np.nan
    #     if units == "nm" and val > 0:
    #         return 9.0 - log10(val)
    #     # you can add more unit handling here (uM, pM, etc.)
    #     return np.nan
    #
    # df["pchembl"] = df.apply(pchembl_from_row, axis=1)
    # df = df[pd.notna(df["pchembl"])].copy()
    #
    # # keep essential columns
    # keep = [
    #     "activity_id", "assay_chembl_id", "assay_type", "assay_confidence_score",
    #     "molecule_chembl_id", "target_chembl_id",
    #     "standard_type", "standard_value", "standard_units", "relation",
    #     "pchembl"
    # ]
    # return df[keep].reset_index(drop=True)

# Example:
# df = fetch_potencies_for_target("CHEMBL203")  # EGFR (human) as example
# print(df.head(), len(df))

# --- 4) (Optional) Attach canonical SMILES for molecules ---
def attach_smiles(df: pd.DataFrame) -> pd.DataFrame:
    mol = new_client.molecule
    ids = df["molecule_chembl_id"].dropna().unique().tolist()
    chunks = [ids[i:i+100] for i in range(0, len(ids), 100)]
    id_to_smiles = {}
    for chunk in chunks:
        mres = mol.filter(molecule_chembl_id__in=chunk).only(
            ["molecule_chembl_id", "molecule_structures"]
        )
        for m in mres:
            smi = None
            if m.get("molecule_structures"):
                smi = m["molecule_structures"].get("canonical_smiles")
            id_to_smiles[m["molecule_chembl_id"]] = smi
    df = df.copy()
    df["canonical_smiles"] = df["molecule_chembl_id"].map(id_to_smiles)
    return df

In [17]:
a = fetch_potencies_for_target("CHEMBL203")
print(type(a))

<class 'list'>


In [19]:
print(a[0])

{'activity_id': 32260, 'assay_chembl_id': 'CHEMBL674637', 'assay_type': 'B', 'molecule_chembl_id': 'CHEMBL68920', 'pchembl_value': '7.39', 'relation': '=', 'standard_type': 'IC50', 'standard_units': 'nM', 'standard_value': '41.0', 'target_chembl_id': 'CHEMBL203', 'type': 'IC50', 'units': 'uM', 'value': '0.041'}


In [29]:
import numbers
print(len(a))
print(len([i for i in a if "pchembl_value" in i.keys()]))
b = [i for i in a if isinstance(i["pchembl_value"], str)]
print(b[0:3])

30166
30166
[{'activity_id': 32260, 'assay_chembl_id': 'CHEMBL674637', 'assay_type': 'B', 'molecule_chembl_id': 'CHEMBL68920', 'pchembl_value': '7.39', 'relation': '=', 'standard_type': 'IC50', 'standard_units': 'nM', 'standard_value': '41.0', 'target_chembl_id': 'CHEMBL203', 'type': 'IC50', 'units': 'uM', 'value': '0.041'}, {'activity_id': 32263, 'assay_chembl_id': 'CHEMBL621151', 'assay_type': 'F', 'molecule_chembl_id': 'CHEMBL68920', 'pchembl_value': '6.52', 'relation': '=', 'standard_type': 'IC50', 'standard_units': 'nM', 'standard_value': '300.0', 'target_chembl_id': 'CHEMBL203', 'type': 'IC50', 'units': 'uM', 'value': '0.3'}, {'activity_id': 32265, 'assay_chembl_id': 'CHEMBL615325', 'assay_type': 'F', 'molecule_chembl_id': 'CHEMBL68920', 'pchembl_value': '5.11', 'relation': '=', 'standard_type': 'IC50', 'standard_units': 'nM', 'standard_value': '7820.0', 'target_chembl_id': 'CHEMBL203', 'type': 'IC50', 'units': 'uM', 'value': '7.82'}]


In [12]:
print(a.shape)
print(a.head())

(0, 11)
Empty DataFrame
Columns: [activity_id, assay_chembl_id, assay_type, assay_confidence_score, molecule_chembl_id, standard_type, standard_value, standard_units, relation, pchembl_value, target_chembl_id]
Index: []
